In [3]:
import pandas as pd
df = pd.read_csv("online_retail_II.csv")
print(df.shape)
print(df.head())
print(df.dtypes)
print(df.isna().sum())

(1067371, 8)
  Invoice StockCode                          Description  Quantity  \
0  489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1  489434    79323P                   PINK CHERRY LIGHTS        12   
2  489434    79323W                  WHITE CHERRY LIGHTS        12   
3  489434     22041         RECORD FRAME 7" SINGLE SIZE         48   
4  489434     21232       STRAWBERRY CERAMIC TRINKET BOX        24   

           InvoiceDate  Price  Customer ID         Country  
0  2009-12-01 07:45:00   6.95      13085.0  United Kingdom  
1  2009-12-01 07:45:00   6.75      13085.0  United Kingdom  
2  2009-12-01 07:45:00   6.75      13085.0  United Kingdom  
3  2009-12-01 07:45:00   2.10      13085.0  United Kingdom  
4  2009-12-01 07:45:00   1.25      13085.0  United Kingdom  
Invoice         object
StockCode       object
Description     object
Quantity         int64
InvoiceDate     object
Price          float64
Customer ID    float64
Country         object
dtype: object
In

In [4]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["Revenue"] = df["Quantity"] * df["Price"]
df["IsReturn"] = df["Invoice"].astype(str).str.startswith("C")

In [5]:
print("Returns:", df["IsReturn"].sum())
print("Exact duplicate rows:", df.duplicated().sum())
print("Price <= 0:", (df["Price"] <= 0).sum())
print("Quantity <= 0 (not returns):", ((df["Quantity"] <= 0) & ~df["IsReturn"]).sum())
print("Date range:", df["InvoiceDate"].min(), "-", df["InvoiceDate"].max())

Returns: 19494
Exact duplicate rows: 34335
Price <= 0: 6207
Quantity <= 0 (not returns): 3457
Date range: 2009-12-01 07:45:00 - 2011-12-09 12:50:00


In [6]:
monthly = (
    df.groupby(df["InvoiceDate"].dt.to_period("M"))
    .agg(
        revenue=("Revenue", "sum"),     
        orders=("Invoice", "nunique"),   
        customers=("Customer ID", "nunique"),  
    )
    .reset_index()
    .rename(columns={"InvoiceDate": "month"})
)
monthly["month"] = monthly["month"].astype(str)
monthly

,month,revenue,orders,customers
0,2009-12,799847.110,2330,1045
1,2010-01,624032.892,1633,786
2,2010-02,533091.426,1969,807
3,2010-03,765848.761,2367,1111
4,2010-04,590580.432,1892,998
5,2010-05,615322.830,2418,1062
6,2010-06,679786.610,2216,1095
7,2010-07,575236.360,2017,988
8,2010-08,656776.340,1877,964
9,2010-09,853650.431,2375,1202


In [7]:
anomalies = df.copy()

anomalies["is_duplicate"] = df.duplicated(keep="first")

anomalies["is_bad_price"] = df["Price"] <= 0

anomalies["is_bad_quantity"] = (df["Quantity"] <= 0) & ~df["IsReturn"]

sales = df[~df["IsReturn"] & (df["Price"] > 0) & (df["Quantity"] > 0)]
z_scores = (sales["Revenue"] - sales["Revenue"].mean()) / sales["Revenue"].std()
anomalies["is_outlier"] = False
anomalies.loc[z_scores[z_scores.abs() > 5].index, "is_outlier"] = True

In [8]:
flag_cols = ["is_duplicate", "is_bad_price", "is_bad_quantity", "is_outlier"]

summary = anomalies[flag_cols].sum().rename("rows").to_frame()
summary["share_pct"] = (summary["rows"] / len(anomalies) * 100).round(2)
summary

,rows,share_pct
is_duplicate,34335,3.22
is_bad_price,6207,0.58
is_bad_quantity,3457,0.32
is_outlier,726,0.07


In [9]:
anomalies["any_flag"] = anomalies[flag_cols].any(axis=1)

anomalies[anomalies["any_flag"]].to_csv("anomalies.csv", index=False)
monthly.to_csv("monthly_kpi.csv", index=False)
summary.reset_index().rename(columns={"index": "rule"}).to_csv("anomaly_summary.csv", index=False)

print("Flagged rows:", anomalies["any_flag"].sum())

Flagged rows: 41060


In [3]:
%run pipeline.py

2026-10-05 00:20:44,975 | INFO | Loading data\online_retail_II.csv.gz
2026-10-05 00:20:46,265 | INFO | Loaded 1067371 rows, 8 columns
2026-10-05 00:20:48,071 | INFO | Done. Flagged 41060 of 1067371 rows
